# Sarcasm Detection
We want to examine news headlines and detect whether or not they are sarcastic. We will start from a pretrained model from the `transformers` library.

## Dataset and Imports

In [ ]:
import opendatasets as ods
#ods.download("https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Dataset URL: https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection


100%|██████████| 3.30M/3.30M [00:04<00:00, 840kB/s]


In [2]:
import torch
import torch.nn as nn
from torch.optim import Adam
from transformers import AutoTokenizer, AutoModel  # Pretrained models
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import pandas as pd 
import numpy as np
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


## Preprocessing Dataset

In [8]:
# The files are in a json format
data_df = pd.read_json('news-headlines-dataset-for-sarcasm-detection/Sarcasm_Headlines_Dataset.json', lines=True)

data_df.dropna(inplace=True)
data_df.drop_duplicates(inplace=True)
data_df.drop(['article_link'], inplace=True, axis=1)
print(data_df.shape)
data_df.head(5)

(26708, 2)


,headline,is_sarcastic
0,former versace store clerk sues over secret 'b...,0
1,the 'roseanne' revival catches up to our thorn...,0
2,mom starting to fear son's web series closest ...,1
3,"boehner just wants wife to listen, not come up...",1
4,j.k. rowling wishes snape happy birthday in th...,0


### Splitting Dataset

In [9]:
# Splitting whole dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(
    np.array(data_df['headline']),
    np.array(data_df['is_sarcastic']),
    test_size=0.3,
    random_state=42
)

# Splitting test dataset into test and validation -> 0.7 train, 0.15 val, 0.15 test
X_test, X_val, y_test, y_val = train_test_split(
    X_test,
    y_test,
    test_size=0.5,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)
print(X_val.shape)

(18695,)
(4006,)
(4007,)


### Create Dataset Class
The tokenizer reduces a string of text into numbers (or *tokens*). The tokenizer basically contains a dictionary that assignes to each word a number, or token. The tokenizer is unique to the model, so we need to use the tokenizer that comes with the model in order to convert our dataset.

Example:
```
Original sentence: "Hello, how are you"
Tokenizer: {'hello':432, 'how': 421, 'are':998, 'you': 811, ...}
Tokens: [432, 421, 998, 811]
```
If padding is applied, the missing values of the tokens array are filled with a 0

In [15]:
class dataset(Dataset):
    def __init__(self, X, y):
        self.X = [tokenizer(
            x, 
            max_length=100,
            truncation=True, # every text longer than 100 gets thrown away
            padding='max_length', # every text shorter than 100 will be padded to be 100 long
            return_tensors='pt'
        ).to(device) for x in X]
        self.y = torch.tensor(y, dtype=torch.float32).to(device)

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [16]:
training_data = dataset(X_train, y_train)
validation_data = dataset(X_val, y_val)
testing_data = dataset(X_test, y_test)

## Building the Model

### Pretrained Model
We are going to start from the `google-bert/bert-base-uncased` model from the `transformers` library. This is one of the most used models for text classification task.

In [11]:
tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-uncased')
bert_model = AutoModel.from_pretrained('google-bert/bert-base-uncased')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 759.83it/s]
[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Hyperparameters

In [17]:
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4

### Dataloader

In [29]:
train_loader = DataLoader(training_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(validation_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(testing_data, batch_size=BATCH_SIZE, shuffle=True)

### Model Class
The `MyModel` class takes in the pretrained `bert` model as an input. Then we add 
- `nn.Dropout` layer: during training, randomly zeroes some of the elements of the input tensor with probability `p`. This is an effective technique for regularization
- `nn.Linear` layers: for the first linear layer, we have $768$ `in_features` as that is the number of `out_features` of the `bert` model, while 384 is an arbitrary number, half of this size
The added layers carry out the classification part

In [ ]:
class MyModel(nn.Module):
    def __init__(self, bert):
        super(MyModel, self).__init__()

        self.bert = bert
        self.dropout = nn.Dropout(p=0.25)
        self.linear1 = nn.Linear(768, 384)
        self.linear2 = nn.Linear(384, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        pooled_output = self.bert(input_ids, attention_mask, return_dict = False)[0][:,0]
        output = self.linear1(pooled_output)
        output = self.dropout(output)
        output = self.linear2(output)
        output = self.sigmoid(output)

        return output

Set the `bert` parameters as unchangeable. This will let the model preserve all its previous knowledge during the training, while the new layers will specialize on our dataset.

In [19]:
for param in bert_model.parameters():
    param.requires_grad = False

In [25]:
model = MyModel(bert_model).to(device)
model

MyModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwis

### Loss and Optimizer

In [26]:
criterion = nn.BCELoss()
optimizer = Adam(model.parameters(), lr=LR)

## Training the Model

In [34]:
from tqdm import tqdm

In [ ]:
total_loss_train_list = []
total_loss_validation_list = []
total_acc_train_list = []
total_acc_validation_list = []

for epoch in range(EPOCHS):
    total_acc_train = 0
    total_loss_train = 0
    total_acc_validation = 0
    total_loss_validation = 0

    for indx, data in tqdm(enumerate(train_loader)):
        inputs, labels = data
        inputs.to(device)
        labels.to(device)

        prediction = model(inputs['input_ids'].squeeze(1), inputs['attention_mask'].squeeze(1)).squeeze(1)
        
        batch_loss = criterion(prediction, labels)

        total_loss_train += batch_loss.item()

        batch_acc = (prediction.round() == labels).sum().item()

        total_acc_train += batch_acc

        batch_loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    with torch.no_grad():
        for indx, data in enumerate(val_loader):
            inputs, labels = data
            inputs.to(device)
            labels.to(device)

            prediction = model(inputs['input_ids'].squeeze(1), inputs['attention_mask'].squeeze(1)).squeeze(1)

            batch_loss = criterion(prediction, labels)

            total_loss_validation += batch_loss.item()

            batch_acc = (prediction.round() == labels).sum().item()

            total_acc_validation += batch_acc

    total_loss_train_list.append(round(total_loss_train/1000, 4))
    total_acc_train_list.append(round(total_acc_train/training_data.__len__() * 100, 4))

    total_loss_validation_list.append(round(total_loss_validation/1000, 4))
    total_acc_validation_list.append(round(total_acc_validation/validation_data.__len__() * 100, 4))

    print(f'''Epoch: {epoch+1} | Train Loss: {round(total_loss_train/1000, 4)} | Train Acc: {round(total_acc_train/training_data.__len__() * 100, 4)} | Val Loss: {round(total_loss_validation/1000, 4)} | Val Acc: {round(total_acc_validation/validation_data.__len__() * 100, 4)}''')



91it [10:25,  6.31s/it]

## Testing the Model

In [ ]:
with torch.no_grad():
    total_loss_test = 0
    total_acc_test = 0
    for indx, data in enumerate(test_loader):
        inputs, labels = data
        inputs.to(device)
        labels.to(device)

        prediction = model(inputs['input_ids'].squeeze(1), inputs['attention_mask'].squeeze(1)).squeeze(1)

        batch_loss = criterion(prediction, labels)

        total_loss_test += batch_loss.item()

        batch_acc = (prediction.round() == labels).sum().item()

        total_acc_test += batch_acc

print(f"Test Acc: {round(total_acc_test/testing_data.__len__() * 100, 4)}")
        

## Visualization

In [ ]:
fig, axs = plt.subplots(nrows = 1, ncols=2, figsize=(15,5))

axs[0].plot(total_loss_train_list, label = 'Training Loss')
axs[0].plot(total_loss_validation_list, label = 'Validation Loss')
axs[0].set_title("Training and Validation loss over epochs")
axs[0].set_xlabel('Epochs')
axs[0].set_ylabel('Loss')
#axs[0].set_ylim([0,2])
axs[0].legend()

axs[1].plot(total_acc_train_list, label = 'Training Accuracy')
axs[1].plot(total_acc_validation_list, label = 'Validation Accuracy')
axs[1].set_title("Training and Validation accuracy over epochs")
axs[1].set_xlabel('Epochs')
axs[1].set_ylabel('Accuracy')
#axs[1].set_ylim([0,100])
axs[1].legend()

plt.show()